# Pashto OCR — CRNN + CTC on Kaggle: printed **and** handwritten

Two-stage training of a CRNN (CNN + BiLSTM + CTC, ~12M params):

1. **Stage 1 — printed:** [`zirak-ai/PashtoOCR`](https://huggingface.co/datasets/zirak-ai/PashtoOCR)
   (10k synthetic paragraph images → ~60k line crops). ~25 min on a T4.
2. **Stage 2 — handwriting:** fine-tune on **KPTI** (Katib's Pashto Text Imagebase,
   [github.com/rahmad77/KPTI](https://github.com/rahmad77/KPTI)) — 17,015 real
   hand-scribed text lines scanned at 300dpi from Pashto books. ~15 min.

The result reads printed / on-screen Pashto **and** katib-style handwritten manuscripts.
Inference accepts **PNG / JPG / any image format / multi-page PDF** (`ocr_file`).

**RTL note:** CTC alignment is monotonic left→right, so every line image is horizontally
flipped during preprocessing (training *and* inference) while the label stays in logical
order. The model simply learns mirrored glyphs — a standard trick for Arabic-script CTC.

**Kaggle setup:** Settings → Accelerator: **GPU T4 x2** (P100 or a single T4 also fine) ·
Internet: **ON**.

> **KPTI license note:** the KPTI data is provided for research; publications using it must
> cite *Ahmad et al., "KPTI: Katib's Pashto Text Imagebase and Deep Learning Benchmark",
> ICFHR 2016.*


In [ ]:
%pip install -q -U "datasets>=2.20" jiwer huggingface_hub pypdfium2


In [ ]:
import ast, json, math, random, shutil, time, unicodedata
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


class CFG:
    dataset_name = "zirak-ai/PashtoOCR"
    kpti_repo    = "https://github.com/rahmad77/KPTI"
    work_dir     = "/kaggle/working"
    kpti_dir     = "/kaggle/working/KPTI"
    out_dir      = "/kaggle/working/pashto-crnn"

    # line crops
    crop_pad   = 3        # px padding around each line bbox
    min_crop_w = 8
    min_crop_h = 6
    img_h      = 48       # every crop resized to this height (aspect preserved)
    max_w      = 1600     # extremely long lines get squeezed to this width
    downsample = 4        # CNN reduces width by 4 -> timesteps = W // 4

    # stage 1: printed (synthetic)
    epochs       = 8
    lr           = 1e-3   # OneCycle peak LR (AdamW)
    batch        = 64
    val_page_mod = 20     # 1/20 pages -> validation (~5%)

    # stage 2: handwriting fine-tune (KPTI)
    ft_epochs    = 12
    ft_lr        = 2e-4
    ft_syn_mix   = 10000  # synthetic lines mixed in so the model keeps its printed skill

    quick_test = False    # True = small subsets / 2 epochs per stage smoke run

device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpus = torch.cuda.device_count()
print(f"device={device}  gpus={n_gpus}")


## 1. Download the printed (synthetic) dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset(CFG.dataset_name, split="test")   # the 10k benchmark is published as split="test"
if CFG.quick_test:
    ds = ds.select(range(300))
print(ds)
print(ds[0]["text"][:200])


## 2. Extract line crops

Each sample is a paragraph image; `lines` holds `{text, bbox:[x, y, width, height]}` per line.
We crop every line, normalize dark-theme crops to dark-text-on-light, convert to grayscale and
**resize to height 48 once at crop time**, then write them to disk. Runs once, ~5–10 min.


In [ ]:
LINES_DIR = Path(CFG.work_dir) / "line_crops"
LINES_DIR.mkdir(parents=True, exist_ok=True)


def clean_text(t):
    t = unicodedata.normalize("NFC", str(t))
    return " ".join(t.split())


def parse_bbox(b):
    if isinstance(b, str):
        b = ast.literal_eval(b)
    x, y, w, h = (float(v) for v in b)
    return x, y, w, h


def maybe_invert(img):
    # Normalize to dark text on light background.
    if np.asarray(img.convert("L")).mean() < 127:
        return ImageOps.invert(img)
    return img


def preprocess_crop(img):
    # -> grayscale, height CFG.img_h, aspect preserved, width capped at CFG.max_w
    g = maybe_invert(img).convert("L")
    w, h = g.size
    new_w = max(CFG.min_crop_w, min(CFG.max_w, round(w * CFG.img_h / h)))
    return g.resize((new_w, CFG.img_h), Image.BILINEAR)


records, skipped = [], 0
for ex in tqdm(ds, desc="cropping lines"):
    img = ex["image"].convert("RGB")
    W, H = img.size
    lines = ex["lines"]
    if isinstance(lines, str):
        lines = ast.literal_eval(lines)
    for j, ln in enumerate(lines):
        text = clean_text(ln["text"])
        if not text:
            skipped += 1
            continue
        try:
            x, y, w, h = parse_bbox(ln["bbox"])
        except Exception:
            skipped += 1
            continue
        x0 = max(0, int(x - CFG.crop_pad)); y0 = max(0, int(y - CFG.crop_pad))
        x1 = min(W, math.ceil(x + w + CFG.crop_pad)); y1 = min(H, math.ceil(y + h + CFG.crop_pad))
        if x1 - x0 < CFG.min_crop_w or y1 - y0 < CFG.min_crop_h:
            skipped += 1
            continue
        crop = preprocess_crop(img.crop((x0, y0, x1, y1)))
        path = LINES_DIR / f"{ex['id']}_{j}.png"
        crop.save(path)
        records.append({"path": str(path), "text": text,
                        "page_id": int(ex["id"]), "width": crop.size[0]})

print(f"line crops: {len(records)}   skipped: {skipped}")


In [ ]:
# Page-level split so no page leaks across train/val.
syn_train = [r for r in records if r["page_id"] % CFG.val_page_mod != 0]
syn_val   = [r for r in records if r["page_id"] % CFG.val_page_mod == 0]

# CTC needs timesteps >= label length (+1 per doubled char). Drop the rare offenders.
def min_frames(t):
    return len(t) + sum(1 for a, b in zip(t, t[1:]) if a == b)

def ctc_fits(r):
    return min_frames(r["text"]) <= r["width"] // CFG.downsample

before = len(syn_train)
syn_train = [r for r in syn_train if ctc_fits(r)]
print(f"printed train lines: {len(syn_train)} (dropped {before - len(syn_train)})   "
      f"val lines: {len(syn_val)}")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 1, figsize=(10, 5))
for ax, r in zip(axes, random.sample(syn_train, 4)):
    ax.imshow(Image.open(r["path"]), cmap="gray"); ax.axis("off")
plt.tight_layout(); plt.show()


## 3. Download KPTI — real handwritten Pashto lines

17,015 hand-scribed text-line images (`.jpg`) with UTF-8 ground truth (`.txt`), already
split into train / valid / test. This is genuine katib handwriting from scanned Pashto
books — the same domain as handwritten manuscript pages.


In [ ]:
import subprocess

if not Path(CFG.kpti_dir).exists():
    subprocess.run(["git", "clone", "--depth", "1", CFG.kpti_repo, CFG.kpti_dir], check=True)


def load_kpti_split(folder):
    recs, missing = [], 0
    for jpg in sorted(Path(folder).glob("*.jpg")):
        gt = jpg.with_suffix(".txt")
        if not gt.exists():
            gt = jpg.with_suffix(".gt.txt")
        if not gt.exists():
            missing += 1
            continue
        text = clean_text(gt.read_text(encoding="utf-8"))
        if not text:
            continue
        with Image.open(jpg) as im:          # header-only read: cheap
            w, h = im.size
        new_w = max(CFG.min_crop_w, min(CFG.max_w, round(w * CFG.img_h / h)))
        recs.append({"path": str(jpg), "text": text, "width": new_w})
    if missing:
        print(f"  {folder}: {missing} images had no ground truth (skipped)")
    return recs


kpti_train = load_kpti_split(Path(CFG.kpti_dir) / "KPTI-TrainData")
kpti_val   = load_kpti_split(Path(CFG.kpti_dir) / "KPTI-ValidData")
kpti_test  = load_kpti_split(Path(CFG.kpti_dir) / "KPTI-TestData")
if CFG.quick_test:
    kpti_train, kpti_val, kpti_test = kpti_train[:1500], kpti_val[:300], kpti_test[:300]

before = len(kpti_train)
kpti_train = [r for r in kpti_train if ctc_fits(r)]
print(f"KPTI train: {len(kpti_train)} (dropped {before - len(kpti_train)})   "
      f"valid: {len(kpti_val)}   test: {len(kpti_test)}")

fig, axes = plt.subplots(3, 1, figsize=(10, 4))
for ax, r in zip(axes, random.sample(kpti_train, 3)):
    ax.imshow(Image.open(r["path"]), cmap="gray"); ax.axis("off")
    print(r["text"])
plt.tight_layout(); plt.show()


## 4. Character vocabulary

Union of the printed and KPTI training texts, so one model (and one output layer) covers
both stages. Index 0 is reserved for the CTC blank.


In [ ]:
charset = sorted({c for r in syn_train for c in r["text"]}
                 | {c for r in kpti_train for c in r["text"]})
char2id = {c: i + 1 for i, c in enumerate(charset)}          # 0 = CTC blank
id2char = {i + 1: c for i, c in enumerate(charset)}
n_classes = len(charset) + 1
print(f"charset size: {len(charset)}  (+1 blank = {n_classes} classes)")
print("".join(charset))


## 5. Model — CRNN

VGG-style CNN (height 48 → 1, width ÷4) → 2-layer BiLSTM → per-timestep classifier.
~12M parameters.


In [ ]:
def conv_bn(cin, cout, pool=None):
    layers = [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
    if pool:
        layers.append(nn.MaxPool2d(pool))
    return layers


class CRNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            *conv_bn(1, 64, pool=(2, 2)),       # 24 x W/2
            *conv_bn(64, 128, pool=(2, 2)),     # 12 x W/4
            *conv_bn(128, 256),
            *conv_bn(256, 256, pool=(2, 1)),    # 6 x W/4
            *conv_bn(256, 512),
            *conv_bn(512, 512, pool=(2, 1)),    # 3 x W/4
            nn.Conv2d(512, 512, (3, 3), padding=(0, 1)),   # 1 x W/4
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        )
        self.rnn = nn.LSTM(512, 256, num_layers=2, bidirectional=True,
                           batch_first=True, dropout=0.1)
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):                 # x: (B, 1, 48, W)
        f = self.cnn(x)                   # (B, 512, 1, W/4)
        f = f.squeeze(2).permute(0, 2, 1) # (B, T, 512)
        out, _ = self.rnn(f)
        return self.fc(out)               # (B, T, n_classes)


model = CRNN(n_classes).to(device)
if n_gpus > 1:
    model = nn.DataParallel(model)
net = model.module if hasattr(model, "module") else model
print(f"parameters: {sum(p.numel() for p in net.parameters())/1e6:.1f}M")


## 6. Dataset, augmentation, width-bucketed batching

Images are flipped horizontally (RTL → LTR for CTC monotonicity). Batches are bucketed by
width so padding is minimal, and batch order is shuffled every epoch. The handwriting stage
uses heavier augmentation (elastic distortion + shear) to mimic pen/scan variation.


In [ ]:
from torchvision import transforms

printed_aug = transforms.Compose([
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.0))], p=0.25),
    transforms.RandomAdjustSharpness(2.0, p=0.25),
])

class SafeElastic:
    # ElasticTransform's internal blur (kernel ~ 8*sigma) reflect-pads by ~4*sigma px,
    # which crashes on very narrow crops -> only apply when the image is wide enough.
    def __init__(self, alpha=25.0, sigma=5.0, p=0.4, min_w=64):
        self.t = transforms.ElasticTransform(alpha=alpha, sigma=sigma, fill=255)
        self.p, self.min_w = p, min_w

    def __call__(self, img):
        if img.size[0] >= self.min_w and random.random() < self.p:
            return self.t(img)
        return img


handwriting_aug = transforms.Compose([
    SafeElastic(alpha=25.0, sigma=5.0, p=0.4),
    transforms.RandomAffine(degrees=1.5, shear=(-6, 6), fill=255),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.0))], p=0.25),
])


def to_tensor(img):
    t = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).unsqueeze(0)
    t = (t - 0.5) / 0.5
    return torch.flip(t, dims=[2])        # hflip: RTL text -> left-to-right frames


class PashtoLineDataset(torch.utils.data.Dataset):
    def __init__(self, recs, aug=None):
        self.recs, self.aug = recs, aug

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, i):
        r = self.recs[i]
        img = Image.open(r["path"])
        if img.mode != "L" or img.size[1] != CFG.img_h:
            img = preprocess_crop(img)    # KPTI images are preprocessed lazily here
        if self.aug is not None:
            img = self.aug(img)
        labels = torch.tensor([char2id[c] for c in r["text"] if c in char2id],
                              dtype=torch.long)
        return {"image": to_tensor(img), "labels": labels, "text": r["text"]}


def collate(batch):
    ws = [b["image"].shape[2] for b in batch]
    W = max(ws)
    x = torch.ones(len(batch), 1, CFG.img_h, W)          # pad with white (=1 after norm)
    for i, b in enumerate(batch):
        x[i, :, :, : ws[i]] = b["image"]
    return {
        "images": x,
        "targets": torch.cat([b["labels"] for b in batch]),
        "target_lengths": torch.tensor([len(b["labels"]) for b in batch]),
        "input_lengths": torch.tensor([w // CFG.downsample for w in ws]),
        "texts": [b["text"] for b in batch],
    }


class BucketBatchSampler(torch.utils.data.Sampler):
    # width-sorted batches, shuffled batch order -> minimal padding, still stochastic
    def __init__(self, widths, batch_size, shuffle):
        order = np.argsort(widths)
        self.batches = [order[i:i + batch_size].tolist()
                        for i in range(0, len(order), batch_size)]
        self.shuffle = shuffle

    def __iter__(self):
        batches = list(self.batches)
        if self.shuffle:
            random.shuffle(batches)
        yield from batches

    def __len__(self):
        return len(self.batches)


def make_loader(recs, aug=None, shuffle=False):
    return torch.utils.data.DataLoader(
        PashtoLineDataset(recs, aug=aug),
        batch_sampler=BucketBatchSampler([r["width"] for r in recs], CFG.batch, shuffle),
        collate_fn=collate, num_workers=2, pin_memory=True)


syn_train_loader = make_loader(syn_train, aug=printed_aug, shuffle=True)
syn_val_loader   = make_loader(syn_val)
print(f"stage-1 train batches/epoch: {len(syn_train_loader)}")


## 7. Training harness + Stage 1 (printed)

AdamW + OneCycle LR, fp16 autocast (CTC loss kept in fp32 for stability), full-validation
CER/WER after every epoch, best checkpoint kept in memory. The same `run_training` is
reused for the handwriting stage.


In [ ]:
import jiwer


def ctc_greedy_decode(best_ids, length, mapping=None):
    mapping = mapping or id2char
    out, prev = [], 0
    for k in best_ids[:length].tolist():
        if k != prev and k != 0:
            out.append(mapping[k])
        prev = k
    return "".join(out)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    refs, hyps = [], []
    for batch in loader:
        with torch.autocast("cuda", enabled=device == "cuda"):
            logits = model(batch["images"].to(device, non_blocking=True))
        best = logits.argmax(2).cpu()
        for ids, L, ref in zip(best, batch["input_lengths"], batch["texts"]):
            refs.append(ref)
            hyps.append(ctc_greedy_decode(ids, L))
    return jiwer.cer(refs, hyps), jiwer.wer(refs, hyps), refs, hyps


def run_training(train_loader, val_loader, epochs, max_lr, tag):
    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=max_lr, total_steps=epochs * len(train_loader), pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")

    best_cer, best_state = float("inf"), None
    for epoch in range(1, epochs + 1):
        model.train()
        t0, running = time.time(), 0.0
        pbar = tqdm(train_loader, desc=f"[{tag}] epoch {epoch}/{epochs}")
        for step, batch in enumerate(pbar, 1):
            images = batch["images"].to(device, non_blocking=True)
            with torch.autocast("cuda", enabled=device == "cuda"):
                logits = model(images)
            log_probs = F.log_softmax(logits.float(), dim=2).permute(1, 0, 2)  # (T, B, C)
            loss = F.ctc_loss(log_probs, batch["targets"],
                              batch["input_lengths"], batch["target_lengths"],
                              blank=0, zero_infinity=True)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item()
            if step % 50 == 0:
                pbar.set_postfix(loss=f"{running/step:.3f}")

        cer, wer, refs, hyps = evaluate(val_loader)
        print(f"[{tag}] epoch {epoch}: train_loss={running/len(train_loader):.3f}  "
              f"val CER={cer:.4f}  WER={wer:.4f}  ({time.time()-t0:.0f}s)")
        for ref, hyp in list(zip(refs, hyps))[:2]:
            print(f"  GT : {ref}\n  OCR: {hyp}")
        if cer < best_cer:
            best_cer = cer
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
            print(f"  new best CER {best_cer:.4f} — checkpoint kept")

    net.load_state_dict(best_state)
    return best_cer


stage1_epochs = 2 if CFG.quick_test else CFG.epochs
cer_printed = run_training(syn_train_loader, syn_val_loader,
                           stage1_epochs, CFG.lr, tag="printed")
print(f"\nStage 1 done — best printed validation CER: {cer_printed:.4f}")


## 8. Export Stage 1 and push to the Hugging Face Hub

In [ ]:
cer, wer, _, _ = evaluate(syn_val_loader)
print(f"printed (best ckpt) — CER: {cer:.4f}   WER: {wer:.4f}")

out = Path(CFG.out_dir); out.mkdir(parents=True, exist_ok=True)
torch.save(net.state_dict(), out / "crnn.pt")
(out / "charset.json").write_text(
    json.dumps({"charset": charset, "img_h": CFG.img_h, "max_w": CFG.max_w,
                "downsample": CFG.downsample}, ensure_ascii=False))
print("saved:", CFG.out_dir)


**One-time token setup:** create a *write* token at
[hf.co/settings/tokens](https://huggingface.co/settings/tokens), then in this Kaggle
notebook go to **Add-ons → Secrets → Add secret**, name it `HF_TOKEN`, paste the token,
and attach it to the notebook. (If the secret is missing you'll be prompted to paste
the token manually.)


In [ ]:
from huggingface_hub import HfApi, create_repo

HF_REPO = "mhalimi3008/pashtoOCR"


def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        import getpass
        return getpass.getpass("Kaggle secret HF_TOKEN not found - paste your HF write token: ")


hf_token = get_hf_token()

MODEL_CARD = '''---
language: ps
license: mit
tags:
- ocr
- crnn
- ctc
- pashto
- handwriting
- image-to-text
metrics:
- cer
- wer
---

# Pashto OCR - CRNN + CTC (printed + handwritten)

Line-level Pashto OCR (CRNN: VGG-style CNN + 2-layer BiLSTM + CTC, ~12M params).

- `crnn.pt` - stage-1 weights: printed / on-screen text
  (trained on [zirak-ai/PashtoOCR](https://huggingface.co/datasets/zirak-ai/PashtoOCR)).
  Printed validation CER = {CER} / WER = {WER}
- `crnn_handwriting.pt` - stage-2 weights: fine-tuned on
  [KPTI](https://github.com/rahmad77/KPTI) (17k real hand-scribed Pashto text lines).
  Reads katib-style handwritten manuscripts as well as printed text. {HW_METRICS}
- `charset.json` - character vocabulary + preprocessing config

## Preprocessing contract

Grayscale, dark-text-on-light (auto-invert dark themes), resized to height 48
(aspect preserved), normalized to [-1, 1], then **horizontally flipped**
(RTL script -> left-to-right CTC frames). Decode with greedy CTC
(collapse repeats, drop blank id 0).

## Usage

```python
import json, torch
from huggingface_hub import hf_hub_download

weights = hf_hub_download("{REPO}", "crnn_handwriting.pt")   # or crnn.pt
cfg = json.loads(open(hf_hub_download("{REPO}", "charset.json")).read())
model = CRNN(len(cfg["charset"]) + 1).eval()      # CRNN class from the training notebook
model.load_state_dict(torch.load(weights, map_location="cpu"))
```

The training notebook (sections 5-6 and 10) contains the full inference code, including
projection-profile line segmentation and PDF support.

If you use the handwriting weights in research, cite:
*Ahmad et al., "KPTI: Katib's Pashto Text Imagebase and Deep Learning Benchmark", ICFHR 2016.*
'''


def write_card(hw_metrics=""):
    (out / "README.md").write_text(
        MODEL_CARD.replace("{CER}", f"{cer:.4f}").replace("{WER}", f"{wer:.4f}")
                  .replace("{HW_METRICS}", hw_metrics).replace("{REPO}", HF_REPO),
        encoding="utf-8")


write_card()
api = HfApi(token=hf_token)
create_repo(HF_REPO, token=hf_token, exist_ok=True)
api.upload_folder(folder_path=str(out), repo_id=HF_REPO,
                  commit_message=f"stage 1 (printed), val CER={cer:.4f}")
print(f"pushed to https://huggingface.co/{HF_REPO}")


## 9. Stage 2 — handwriting fine-tune on KPTI

Continues training the same model on real handwritten lines, mixed with a slice of the
synthetic data so it keeps its printed-text skill. Lower LR, heavier augmentation.
Published benchmarks on KPTI report ~9–10% CER (MDLSTM, ICFHR 2016) — expect this
CRNN to land in that neighborhood; handwriting is intrinsically harder than print.


In [ ]:
ft_epochs = 2 if CFG.quick_test else CFG.ft_epochs
ft_train = kpti_train + random.sample(syn_train, min(CFG.ft_syn_mix, len(syn_train)))
random.shuffle(ft_train)

ft_train_loader = make_loader(ft_train, aug=handwriting_aug, shuffle=True)
kpti_val_loader = make_loader(kpti_val)
print(f"stage-2 train lines: {len(ft_train)} ({len(kpti_train)} handwritten + synthetic mix)")

cer_hw = run_training(ft_train_loader, kpti_val_loader,
                      ft_epochs, CFG.ft_lr, tag="handwriting")

kpti_test_loader = make_loader(kpti_test)
test_cer, test_wer, refs, hyps = evaluate(kpti_test_loader)
print(f"\nKPTI TEST — CER: {test_cer:.4f}   WER: {test_wer:.4f}")
for ref, hyp in random.sample(list(zip(refs, hyps)), 3):
    print(f"GT : {ref}\nOCR: {hyp}\n" + "-" * 60)


In [ ]:
# Save + push the handwriting model
torch.save(net.state_dict(), out / "crnn_handwriting.pt")
write_card(hw_metrics=f"KPTI test CER = {test_cer:.4f} / WER = {test_wer:.4f}.")
api.upload_folder(folder_path=str(out), repo_id=HF_REPO,
                  commit_message=f"stage 2 (handwriting), KPTI test CER={test_cer:.4f}")
shutil.make_archive(f"{CFG.work_dir}/pashto_ocr_model", "zip", CFG.out_dir)
print(f"pushed to https://huggingface.co/{HF_REPO}  (+ pashto_ocr_model.zip in Output tab)")


### Test: reload from the Hub and OCR handwritten samples

Round-trip check — downloads the published model straight back from
`mhalimi3008/pashtoOCR` and runs it on real handwritten KPTI test lines.


In [ ]:
from huggingface_hub import hf_hub_download

hub_weights = hf_hub_download(HF_REPO, "crnn_handwriting.pt", token=hf_token)
hub_cfg = json.loads(Path(hf_hub_download(HF_REPO, "charset.json", token=hf_token)).read_text())
hub_id2char = {i + 1: c for i, c in enumerate(hub_cfg["charset"])}

hub_model = CRNN(len(hub_cfg["charset"]) + 1).to(device).eval()
hub_model.load_state_dict(torch.load(hub_weights, map_location=device))

for r in random.sample(kpti_test, 3):
    t = to_tensor(preprocess_crop(Image.open(r["path"]))).unsqueeze(0).to(device)
    with torch.no_grad():
        best = hub_model(t).argmax(2)[0].cpu()
    print("GT :", r["text"])
    print("OCR:", ctc_greedy_decode(best, best.shape[0], mapping=hub_id2char))
    print("-" * 60)


## 10. Inference — extract text from any image or PDF

`ocr_file(path)` accepts PNG / JPG / any image format / **multi-page PDF**.
Single-line crops are read directly; multi-line pages are first split into lines with a
horizontal projection-profile segmenter.


In [ ]:
import cv2

model.eval()


def binarize(gray):
    # Flatten photo illumination (shadows/gradients), then Otsu -> ink mask (255 = ink).
    k = max(15, (min(gray.shape) // 20) | 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
    bg = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, kernel)
    norm = cv2.divide(gray, cv2.max(bg, 1), scale=255)
    thr = cv2.threshold(norm, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    return norm, thr


def strip_rules_and_borders(thr):
    # Remove ruled notebook lines, page borders and decorative bars.
    H, W = thr.shape
    horiz = cv2.morphologyEx(
        thr, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (max(20, W // 3), 1)))
    # only subtract THIN horizontal structures (true ruled lines, not bold text strokes)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(horiz)
    thin = np.zeros_like(horiz)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_HEIGHT] <= 6:
            thin[lab == i] = 255
    vert = cv2.morphologyEx(
        thr, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(20, H // 3))))
    thr = cv2.subtract(cv2.subtract(thr, thin), vert)
    return cv2.morphologyEx(thr, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))


def segment_lines(pil_img, pad=4):
    # Split a page into text-line crops: binarize -> drop rules/borders -> smear
    # words into line blobs -> group connected components into lines (top-to-bottom).
    rgb = pil_img.convert("RGB")
    gray = np.asarray(rgb.convert("L"))
    if gray.mean() < 127:
        gray = 255 - gray
    norm, thr = binarize(gray)
    thr = strip_rules_and_borders(thr)
    H, W = thr.shape

    # median stroke-blob height ~ text height scale
    n, _, stats, _ = cv2.connectedComponentsWithStats(thr)
    hs = [stats[i, cv2.CC_STAT_HEIGHT] for i in range(1, n)
          if stats[i, cv2.CC_STAT_AREA] >= 4 and stats[i, cv2.CC_STAT_HEIGHT] < H // 2]
    if not hs:
        return [Image.fromarray(norm)]
    text_h = max(6, int(np.median(hs)))

    # smear horizontally so words merge into line blobs (small vertical tolerance)
    smeared = cv2.dilate(thr, cv2.getStructuringElement(
        cv2.MORPH_RECT, (max(3, text_h * 2), max(1, text_h // 4))))
    n, _, stats, _ = cv2.connectedComponentsWithStats(smeared)
    boxes = [stats[i, :4] for i in range(1, n)
             if stats[i, cv2.CC_STAT_AREA] >= text_h * text_h
             and stats[i, cv2.CC_STAT_HEIGHT] >= max(6, text_h // 2)]

    # group blobs whose vertical centers fall in the same band into one line
    boxes.sort(key=lambda b: b[1] + b[3] / 2)
    lines = []
    for x, y, w, h in boxes:
        cy = y + h / 2
        if lines and cy < lines[-1]["y1"] and (
                min(lines[-1]["y1"], y + h) - max(lines[-1]["y0"], y)
                > 0.5 * min(h, lines[-1]["y1"] - lines[-1]["y0"])):
            ln = lines[-1]
            ln["x0"], ln["y0"] = min(ln["x0"], x), min(ln["y0"], y)
            ln["x1"], ln["y1"] = max(ln["x1"], x + w), max(ln["y1"], y + h)
        else:
            lines.append({"x0": x, "y0": y, "x1": x + w, "y1": y + h})

    # Reference line height = width-weighted median, so wide real text lines set the
    # scale and small noise blobs don't. Then drop far-too-short boxes (specks,
    # leftover rule fragments) and far-too-tall ones (pen photos, graphics).
    hs = np.array([ln["y1"] - ln["y0"] for ln in lines], dtype=float)
    ws = np.array([ln["x1"] - ln["x0"] for ln in lines], dtype=float)
    order = np.argsort(hs)
    cum = np.cumsum(ws[order])
    med_line_h = hs[order][np.searchsorted(cum, cum[-1] / 2)]
    lines = [ln for ln, h in zip(lines, hs)
             if 0.35 * med_line_h <= h <= 3.5 * med_line_h]
    if not lines:
        return [Image.fromarray(norm)]

    crops = []
    for ln in sorted(lines, key=lambda l: l["y0"]):
        x0, y0 = max(0, ln["x0"] - pad), max(0, ln["y0"] - pad)
        x1, y1 = min(W, ln["x1"] + pad), min(H, ln["y1"] + pad)
        crops.append(Image.fromarray(norm[y0:y1, x0:x1]))
    return crops


@torch.no_grad()
def ocr_image(path_or_img):
    # OCR one image (any format). Returns recognized text, lines joined with newlines.
    img = path_or_img if isinstance(path_or_img, Image.Image) else Image.open(path_or_img)
    out_lines = []
    for crop in segment_lines(img):
        t = to_tensor(preprocess_crop(crop)).unsqueeze(0).to(device)
        with torch.autocast("cuda", enabled=device == "cuda"):
            logits = model(t)
        best = logits.argmax(2)[0].cpu()
        out_lines.append(ctc_greedy_decode(best, best.shape[0]))
    return "\n".join(t.strip() for t in out_lines)


def load_pages(path, dpi=300):
    # PDF -> list of page images; anything else -> [image]
    p = str(path)
    if p.lower().endswith(".pdf"):
        import pypdfium2 as pdfium
        return [page.render(scale=dpi / 72).to_pil() for page in pdfium.PdfDocument(p)]
    return [Image.open(p)]


def ocr_file(path):
    # OCR a PNG / JPG / any image / multi-page PDF. Pages separated by blank lines.
    return "\n\n".join(ocr_image(pg) for pg in load_pages(path))


# Demo: one printed line and one handwritten line
for r in [random.choice(syn_val), random.choice(kpti_test)]:
    print("GT :", r["text"])
    print("OCR:", ocr_image(r["path"]))
    print("-" * 60)

In [ ]:
# OCR your own file: upload it via  Add Data / upload, then point at it here.
# Works with .png, .jpg, .jpeg, .bmp, .tiff, ... and multi-page .pdf
# print(ocr_file("/kaggle/input/your-upload/manuscript.pdf"))
# print(ocr_file("/kaggle/input/your-upload/page.png"))


## Using the trained model later

Straight from your Hub repo (no Kaggle needed):

```python
import json, torch
from huggingface_hub import hf_hub_download
from PIL import Image

REPO = "mhalimi3008/pashtoOCR"
weights = hf_hub_download(REPO, "crnn_handwriting.pt")   # or "crnn.pt" for printed-only
cfg = json.loads(open(hf_hub_download(REPO, "charset.json")).read())
charset = cfg["charset"]
id2char = {i + 1: c for i, c in enumerate(charset)}

model = CRNN(len(charset) + 1)                 # copy the CRNN class from section 5
model.load_state_dict(torch.load(weights, map_location="cpu"))
model.eval()
```
(then reuse `preprocess_crop`, `to_tensor`, `ctc_greedy_decode`, `segment_lines`,
`ocr_image`, `ocr_file` from sections 2, 6, 7 and 10).

## Honest limitations — "any handwriting"

- **Katib-style manuscript / book handwriting:** covered by KPTI fine-tuning — this is the
  best openly available handwritten-Pashto data today.
- **Arbitrary personal handwriting** (notes, letters, forms): partially covered at best.
  The bigger PHTI dataset (36k lines, 400 writers, IEEE Access 2022) is **not** publicly
  downloadable — it must be requested from its authors. For a specific writer or document
  style, transcribing even 1–2k of your own lines and running another `run_training`
  round at `ft_lr` gives the largest accuracy jump available.
- **Segmentation:** the projection-profile splitter assumes roughly horizontal lines.
  For skewed or complex layouts, plug a text detector (CRAFT / PaddleOCR DBNet) in front
  of `ocr_image`.
